# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/` (local only, gitignored) — flat directory, cohort determined from each
file's own content (`deployment_mode`), not from which subdirectory it lives in.

In [1]:
import warnings
import plotly.io as pio

from analysis.utils.parser_v2 import load_data
from analysis.pilot.figures import (
    fig_completion_status,
    fig_demographics,
    fig_trial_duration,
    fig_moves,
    fig_reliability,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_pairwise_distance_distribution,
    fig_reliability_vs_distance,
    fig_move_temporal_profile,
    trial_ks_distance,
)

pio.renderers.default = "browser"

## 0. Load data

In [2]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = load_data("data")

df_participants = data["participants"]
df_trials = data["trials"].merge(
    df_participants[["participant_id", "task_version", "sort_area_width", "sort_area_height"]],
    on="participant_id", how="left",
)

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nParticipants dataframe: {df_participants.shape[0]} rows x {df_participants.shape[1]} cols")
print(f"Trials dataframe: {df_trials.shape[0]} rows x {df_trials.shape[1]} cols")
print(df_participants["status"].value_counts().to_string())


Participants dataframe: 101 rows x 26 cols
Trials dataframe: 1546 rows x 20 cols
status
full data          75
revoked consent    16
missing data        8
screened out        2


Versions present in this dataset (exact minor version, not merged): **v1.0**, **v2.0**, **v3.0**,
**v3.06**, **v4.0**. Figures below merge sub-versions onto their major version (v3.0/v3.06 -> "v3")
for display; the `summary` table right below keeps the full minor-version breakdown. This cell is
manually maintained -- update it if a new minor version is added to the data.

In [3]:
main_trials = df_trials[~df_trials["is_catch"]].assign(
    rt_s=lambda d: d["rt"] / 1000,
    qc_flag_int=lambda d: d["qc_flag"].astype(int),
)

per_version_trials = (
    main_trials
    .groupby("task_version")
    .agg(
        rt_s_mean=("rt_s", "mean"),
        rt_s_median=("rt_s", "median"),
        rt_s_sd=("rt_s", "std"),
        rt_s_min=("rt_s", "min"),
        rt_s_max=("rt_s", "max"),
        moves_mean=("num_moves", "mean"),
        moves_median=("num_moves", "median"),
        moves_sd=("num_moves", "std"),
        qc_flag_rate=("qc_flag_int", "mean"),
    )
)

reliability_by_version = (
    df_trials[df_trials["reliability"].notna()]
    .groupby("task_version")["reliability"]
    .median()
    .rename("reliability")
)

participant_counts = (
    df_participants
    .dropna(subset=["task_version"])
    .groupby("task_version")
    .agg(
        total_N=("participant_id", "count"),
        analytic_N=("status", lambda s: int((s == "full data").sum())),
    )
)

summary = (
    participant_counts
    .join(per_version_trials)
    .join(reliability_by_version)
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary

task_version,v1,v2,v3,v3.06,v4
total_N,15.00,10.00,12.00,14.00,31.00
analytic_N,15.00,10.00,11.00,11.00,28.00
rt_s_mean,111.19,73.86,78.53,81.83,85.60
rt_s_median,90.15,65.36,64.86,68.58,73.59
rt_s_sd,93.77,22.27,40.64,37.22,41.01
rt_s_min,22.26,60.59,60.04,60.09,60.24
rt_s_max,926.99,227.23,395.10,407.50,443.92
moves_mean,36.11,28.06,28.49,25.37,28.56
moves_median,26.00,27.00,26.50,23.00,26.00
moves_sd,24.02,7.96,12.57,10.18,11.86


`total_N` = all participants with a resolvable session file for that version (participants who
revoked consent or have no file at all can't be attributed to any version, so they're outside this
table entirely -- see `df_participants["status"]` for the full picture). `analytic_N` = the subset
with `status == "full data"` (no screen-out, no missing data). `reliability` = median test-retest
Spearman r across that version's repeat trials (NaN for v1/v2, which predate the trial-repeat
mechanism).

## 1. Completion status

In [4]:
fig_completion_status(df_participants).show()

## 2. Trial duration

In [5]:
fig_trial_duration(df_trials).show()

## 3. Number of moves

In [6]:
fig_moves(df_trials).show()

## 4. Reliability

In [7]:
fig_reliability(df_trials).show()

## 5. Pairwise distance distribution

In [8]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

from analysis.pilot.simulate_null_distances import simulate as _sim_null

# Shared null: same K as images_per_trial, enough trials for a stable reference
null_distances = _sim_null(num_dots=20, num_trials=1000, seed=42)
print(f"Null: {len(null_distances):,} distances  mean={null_distances.mean():.3f}  sd={null_distances.std():.3f}")

fig_pairwise_distance_distribution(df_trials, null_distribution=null_distances).show()

Null: 190,000 distances  mean=0.369  sd=0.175


## 6. Reliability vs. distance

In [9]:
fig_reliability_vs_distance(df_trials).show()

## 7. Trial duration vs. number of moves

In [10]:
fig_duration_vs_moves(df_trials).show()

## 8. Temporal engagement

Do subjects work throughout the trial, or front-load moves and sit idle?

- **Left**: cumulative move fraction vs. time fraction within trial. Diagonal = uniform activity; curve bending top-left = front-loading.
- **Right**: average move rate (moves/s, 5 s bins) over absolute time. Dashed line = 60 s (V2 timer unlock).

In [11]:
fig_move_temporal_profile(df_trials).show()

## 9. Within-subject variability and reliability

In [12]:
fig_within_subject_variability(df_trials).show()

## 10. Participant demographics

In [13]:
fig_demographics(df_participants).show()

## 11. Cohort comparisons

Pairwise Mann-Whitney U tests between task versions, on per-subject aggregates (one value
per subject, avoiding pseudo-replication across trials). With more than two cohorts present,
every pairwise comparison within a metric is run and p-values are corrected across that
family (default Bonferroni; configurable via `mwu_family(..., correction=...)` -- any method
string accepted by `statsmodels.stats.multitest.multipletests` works, e.g. `"holm"`, `"fdr_bh"`).

Six comparisons, ordered to match the figures above:
1. Trial duration (s)
2. Moves per trial
3. KS distance from null -- per-trial D, averaged per subject
4. Idle tail fraction -- (RT - t_last_move) / RT
5. SNR (sigma_d / mean|delta d|) -- **v1 vs v2 only**, see note before that cell. Moved last: with
   the trial-repeat mechanism now covering v3+, this v1/v2-specific measure is the least central
   comparison in this section.
6. Reliability (Spearman r) across v3+ minor versions -- new, see note before that cell.

Effect size is rank-biserial *r* = 1 - 2U / (n_A x n_B); |r| >= 0.5 is conventionally large.

In [14]:
import json
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


def mwu_compare(df, value_col, group_a, group_b, group_col="task_version"):
    """Pairwise Mann-Whitney U between two specified groups (no group-count assumption)."""
    a = df.loc[df[group_col] == group_a, value_col].dropna().astype(float).values
    b = df.loc[df[group_col] == group_b, value_col].dropna().astype(float).values
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * U / (len(a) * len(b))
    return {
        "comparison":  f"v{group_a:g} vs v{group_b:g}",
        "N (A)":       len(a),                              "N (B)":       len(b),
        "median (A)":  round(float(np.median(a)), 3),       "median (B)":  round(float(np.median(b)), 3),
        "mean (A)":    round(float(np.mean(a)), 3),         "mean (B)":    round(float(np.mean(b)), 3),
        "std (A)":     round(float(np.std(a, ddof=1)), 3),  "std (B)":     round(float(np.std(b, ddof=1)), 3),
        "U":           round(U, 1),
        "p_raw":       p,
        "r":           round(r, 3),
    }


def mwu_family(df, value_col, group_col="task_version", correction="bonferroni", alpha=0.05):
    """
    Pairwise Mann-Whitney U for every pair of groups in group_col, with a multiple-
    comparison correction applied across the family. `correction` accepts any method
    string supported by statsmodels.stats.multitest.multipletests, e.g. "bonferroni",
    "holm", "fdr_bh", "fdr_by", "sidak", "holm-sidak", ...
    """
    groups = sorted(df[group_col].dropna().unique())
    rows = [mwu_compare(df, value_col, a, b, group_col) for a, b in combinations(groups, 2)]
    result = pd.DataFrame(rows).set_index("comparison")
    _, p_corrected, _, _ = multipletests(result["p_raw"], alpha=alpha, method=correction)
    result["p_raw"] = result["p_raw"].round(4)
    result["p_corrected"] = p_corrected.round(4)
    return result[["N (A)", "N (B)", "median (A)", "median (B)", "mean (A)", "mean (B)",
                    "std (A)", "std (B)", "U", "p_corrected", "p_raw", "r"]]


def print_bottom_line(result, alpha=0.05):
    """One-line plain-English summary per pairwise comparison."""
    for comparison, row in result.iterrows():
        sig = "significant" if row["p_corrected"] < alpha else "not significant"
        print(f"  {comparison}: U={row['U']:.1f}  p_corrected={row['p_corrected']:.4f} ({sig}, "
              f"p_raw={row['p_raw']:.4f})  r={row['r']:.3f}")

### Test 1 — Trial duration

In [15]:
subj_rt = (
    df_trials.assign(rt_s=df_trials["rt"] / 1000)
    .groupby(["participant_id", "task_version"])["rt_s"]
    .mean()
    .reset_index()
)

result_rt = mwu_family(subj_rt, "rt_s", correction="bonferroni")
print_bottom_line(result_rt)
result_rt

  v1 vs v2: U=117.0  p_corrected=0.2133 (not significant, p_raw=0.0213)  r=-0.560
  v1 vs v3: U=119.0  p_corrected=0.6171 (not significant, p_raw=0.0617)  r=-0.442
  v1 vs v3.06: U=104.0  p_corrected=1.0000 (not significant, p_raw=0.2758)  r=-0.261
  v1 vs v4: U=274.0  p_corrected=1.0000 (not significant, p_raw=0.2429)  r=-0.218
  v2 vs v3: U=47.0  p_corrected=1.0000 (not significant, p_raw=0.5974)  r=0.145
  v2 vs v3.06: U=22.0  p_corrected=0.2210 (not significant, p_raw=0.0221)  r=0.600
  v2 vs v4: U=60.0  p_corrected=0.0518 (not significant, p_raw=0.0052)  r=0.600
  v3 vs v3.06: U=25.0  p_corrected=0.2155 (not significant, p_raw=0.0215)  r=0.587
  v3 vs v4: U=67.0  p_corrected=0.0412 (significant, p_raw=0.0041)  r=0.594
  v3.06 vs v4: U=149.0  p_corrected=1.0000 (not significant, p_raw=0.6483)  r=0.097


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,95.845,61.402,100.742,65.655,56.173,11.993,117.0,0.2133,0.0213,-0.560
v1 vs v3,15,11,95.845,62.004,100.742,69.376,56.173,21.322,119.0,0.6171,0.0617,-0.442
v1 vs v3.06,15,11,95.845,73.799,100.742,75.630,56.173,9.799,104.0,1.0000,0.2758,-0.261
v1 vs v4,15,30,95.845,75.004,100.742,79.592,56.173,16.067,274.0,1.0000,0.2429,-0.218
v2 vs v3,10,11,61.402,62.004,65.655,69.376,11.993,21.322,47.0,1.0000,0.5974,0.145
v2 vs v3.06,10,11,61.402,73.799,65.655,75.630,11.993,9.799,22.0,0.2210,0.0221,0.600
v2 vs v4,10,30,61.402,75.004,65.655,79.592,11.993,16.067,60.0,0.0518,0.0052,0.600
v3 vs v3.06,11,11,62.004,73.799,69.376,75.630,21.322,9.799,25.0,0.2155,0.0215,0.587
v3 vs v4,11,30,62.004,75.004,69.376,79.592,21.322,16.067,67.0,0.0412,0.0041,0.594


### Test 2 — Moves per trial

In [16]:
subj_moves = (
    df_trials
    .groupby(["participant_id", "task_version"])["num_moves"]
    .mean()
    .reset_index()
)

result_moves = mwu_family(subj_moves, "num_moves", correction="bonferroni")
print_bottom_line(result_moves)
result_moves

  v1 vs v2: U=82.0  p_corrected=1.0000 (not significant, p_raw=0.7182)  r=-0.093
  v1 vs v3: U=91.0  p_corrected=1.0000 (not significant, p_raw=0.6779)  r=-0.103
  v1 vs v3.06: U=102.0  p_corrected=1.0000 (not significant, p_raw=0.3239)  r=-0.236
  v1 vs v4: U=243.5  p_corrected=1.0000 (not significant, p_raw=0.6647)  r=-0.082
  v2 vs v3: U=57.0  p_corrected=1.0000 (not significant, p_raw=0.9159)  r=-0.036
  v2 vs v3.06: U=66.0  p_corrected=1.0000 (not significant, p_raw=0.4595)  r=-0.200
  v2 vs v4: U=136.0  p_corrected=1.0000 (not significant, p_raw=0.6733)  r=0.093
  v3 vs v3.06: U=70.0  p_corrected=1.0000 (not significant, p_raw=0.5544)  r=-0.157
  v3 vs v4: U=152.0  p_corrected=1.0000 (not significant, p_raw=0.7130)  r=0.079
  v3.06 vs v4: U=126.0  p_corrected=1.0000 (not significant, p_raw=0.2572)  r=0.236


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,24.333,24.875,32.417,25.200,17.559,5.386,82.0,1.0,0.7182,-0.093
v1 vs v3,15,11,24.333,25.917,32.417,25.973,17.559,9.607,91.0,1.0,0.6779,-0.103
v1 vs v3.06,15,11,24.333,22.167,32.417,23.511,17.559,7.368,102.0,1.0,0.3239,-0.236
v1 vs v4,15,30,24.333,25.980,32.417,27.006,17.559,8.431,243.5,1.0,0.6647,-0.082
v2 vs v3,10,11,24.875,25.917,25.200,25.973,5.386,9.607,57.0,1.0,0.9159,-0.036
v2 vs v3.06,10,11,24.875,22.167,25.200,23.511,5.386,7.368,66.0,1.0,0.4595,-0.200
v2 vs v4,10,30,24.875,25.980,25.200,27.006,5.386,8.431,136.0,1.0,0.6733,0.093
v3 vs v3.06,11,11,25.917,22.167,25.973,23.511,9.607,7.368,70.0,1.0,0.5544,-0.157
v3 vs v4,11,30,25.917,25.980,25.973,27.006,9.607,8.431,152.0,1.0,0.7130,0.079


### Test 3 — KS distance from null

D is computed per trial against the shared null (see section 5), then averaged per subject —
the same aggregation as `fig_pairwise_distance_distribution` above.

In [17]:
# null_distances defined in the figures section above (section 5)
df_trials_ks = df_trials.assign(
    ks_D=df_trials["pairwise_distances"].apply(trial_ks_distance, null_distribution=null_distances)
)
subj_ks = (
    df_trials_ks
    .groupby(["participant_id", "task_version"])["ks_D"]
    .mean()
    .reset_index()
)

result_ks = mwu_family(subj_ks, "ks_D", correction="bonferroni")
print_bottom_line(result_ks)
result_ks

  v1 vs v2: U=32.0  p_corrected=0.1840 (not significant, p_raw=0.0184)  r=0.573
  v1 vs v3: U=87.0  p_corrected=1.0000 (not significant, p_raw=0.8355)  r=-0.055
  v1 vs v3.06: U=61.0  p_corrected=1.0000 (not significant, p_raw=0.2758)  r=0.261
  v1 vs v4: U=354.0  p_corrected=0.0198 (significant, p_raw=0.0020)  r=-0.573
  v2 vs v3: U=81.0  p_corrected=0.7255 (not significant, p_raw=0.0725)  r=-0.473
  v2 vs v3.06: U=66.0  p_corrected=1.0000 (not significant, p_raw=0.4597)  r=-0.200
  v2 vs v4: U=264.0  p_corrected=0.0039 (significant, p_raw=0.0004)  r=-0.760
  v3 vs v3.06: U=43.0  p_corrected=1.0000 (not significant, p_raw=0.2643)  r=0.289
  v3 vs v4: U=271.0  p_corrected=0.0191 (significant, p_raw=0.0019)  r=-0.642
  v3.06 vs v4: U=283.0  p_corrected=0.0055 (significant, p_raw=0.0005)  r=-0.715


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,0.265,0.296,0.275,0.304,0.036,0.039,32.0,0.1840,0.0184,0.573
v1 vs v3,15,11,0.265,0.262,0.275,0.286,0.036,0.056,87.0,1.0000,0.8355,-0.055
v1 vs v3.06,15,11,0.265,0.276,0.275,0.312,0.036,0.079,61.0,1.0000,0.2758,0.261
v1 vs v4,15,30,0.265,0.231,0.275,0.240,0.036,0.043,354.0,0.0198,0.0020,-0.573
v2 vs v3,10,11,0.296,0.262,0.304,0.286,0.039,0.056,81.0,0.7255,0.0725,-0.473
v2 vs v3.06,10,11,0.296,0.276,0.304,0.312,0.039,0.079,66.0,1.0000,0.4597,-0.200
v2 vs v4,10,30,0.296,0.231,0.304,0.240,0.039,0.043,264.0,0.0039,0.0004,-0.760
v3 vs v3.06,11,11,0.262,0.276,0.286,0.312,0.056,0.079,43.0,1.0000,0.2643,0.289
v3 vs v4,11,30,0.262,0.231,0.286,0.240,0.056,0.043,271.0,0.0191,0.0019,-0.642


### Test 4 — Idle tail fraction

In [18]:
def _idle_tail_fraction(row):
    """Fraction of trial time after the last move: (RT - t_last) / RT."""
    try:
        moves = json.loads(row["moves"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    ts = [m["t"] for m in moves if isinstance(m.get("t"), (int, float))]
    if not ts or pd.isna(row["rt"]) or row["rt"] <= 0:
        return np.nan
    return (row["rt"] - max(ts)) / row["rt"]


subj_idle = (
    df_trials
    .assign(idle_tail=df_trials.apply(_idle_tail_fraction, axis=1))
    .groupby(["participant_id", "task_version"])["idle_tail"]
    .mean()
    .reset_index()
)

result_idle = mwu_family(subj_idle, "idle_tail", correction="bonferroni")
print_bottom_line(result_idle)
result_idle

  v1 vs v2: U=29.0  p_corrected=0.1161 (not significant, p_raw=0.0116)  r=0.613
  v1 vs v3: U=29.0  p_corrected=0.0595 (not significant, p_raw=0.0059)  r=0.648
  v1 vs v3.06: U=51.0  p_corrected=1.0000 (not significant, p_raw=0.1076)  r=0.382
  v1 vs v4: U=88.0  p_corrected=0.0101 (significant, p_raw=0.0010)  r=0.609
  v2 vs v3: U=44.0  p_corrected=1.0000 (not significant, p_raw=0.4597)  r=0.200
  v2 vs v3.06: U=61.0  p_corrected=1.0000 (not significant, p_raw=0.6985)  r=-0.109
  v2 vs v4: U=145.0  p_corrected=1.0000 (not significant, p_raw=0.8882)  r=0.033
  v3 vs v3.06: U=78.0  p_corrected=1.0000 (not significant, p_raw=0.2643)  r=-0.289
  v3 vs v4: U=196.0  p_corrected=1.0000 (not significant, p_raw=0.3695)  r=-0.188
  v3.06 vs v4: U=144.0  p_corrected=1.0000 (not significant, p_raw=0.5464)  r=0.127


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,0.042,0.068,0.044,0.105,0.018,0.098,29.0,0.1161,0.0116,0.613
v1 vs v3,15,11,0.042,0.116,0.044,0.173,0.018,0.164,29.0,0.0595,0.0059,0.648
v1 vs v3.06,15,11,0.042,0.057,0.044,0.103,0.018,0.099,51.0,1.0000,0.1076,0.382
v1 vs v4,15,30,0.042,0.079,0.044,0.108,0.018,0.091,88.0,0.0101,0.0010,0.609
v2 vs v3,10,11,0.068,0.116,0.105,0.173,0.098,0.164,44.0,1.0000,0.4597,0.200
v2 vs v3.06,10,11,0.068,0.057,0.105,0.103,0.098,0.099,61.0,1.0000,0.6985,-0.109
v2 vs v4,10,30,0.068,0.079,0.105,0.108,0.098,0.091,145.0,1.0000,0.8882,0.033
v3 vs v3.06,11,11,0.116,0.057,0.173,0.103,0.164,0.099,78.0,1.0000,0.2643,-0.289
v3 vs v4,11,30,0.116,0.079,0.173,0.108,0.164,0.091,196.0,1.0000,0.3695,-0.188


### Test 5 — SNR (v1 vs v2 only)

**Why v3+ is excluded, not generalized:** v1/v2 reliability is measured from individual image
pairs that incidentally recur across distinct trials (a side effect of the old
`unique_images_per_subject` design — different surrounding images each time). v3+ instead
repeats whole trials verbatim (`repeat_of_trial`), holding context constant. These two measures
convolve different sources of variance — v1/v2 includes context effects, v3+ is closer to pure
response noise — so they are not comparable in absolute magnitude (see the subtitle on the
within-subject variability figure above). v3+ is therefore dropped here rather than silently
mixed into the v1-vs-v2 comparison.

**Test 6 below is the "Test 3′" envisioned when this note was first written**: now that v4 also
uses the trial-repeat mechanism, Test 6 compares reliability across v3+ minor versions directly
using the precomputed `reliability` column, without needing this SNR measure at all.

In [19]:
from collections import defaultdict

from analysis.utils.parser import parse_pairwise_distances


def _subject_snr(df_s):
    """v1/v2-only reliability measure: incidentally-repeated image pairs (see note above)."""
    pair_obs = defaultdict(list)
    for pw_json in df_s["pairwise_distances"]:
        for pair, dist in parse_pairwise_distances(pw_json).items():
            pair_obs[pair].append(dist)
    d1, d2 = [], []
    for obs in pair_obs.values():
        if len(obs) >= 2:
            d1.append(obs[0])
            d2.append(obs[1])
    if not d1:
        return np.nan
    all_dists = [d for pw_json in df_s["pairwise_distances"]
                 for d in parse_pairwise_distances(pw_json).values()]
    sigma_d = float(np.std(all_dists)) if len(all_dists) > 1 else 1.0
    mean_abs_diff = float(np.mean(np.abs(np.array(d1) - np.array(d2))))
    return sigma_d / mean_abs_diff if mean_abs_diff > 0 else np.nan


df_trials_v1v2 = df_trials[df_trials["task_version"].isin([1.0, 2.0])]

subj_snr = (
    df_trials_v1v2
    .groupby(["participant_id", "task_version"])
    .apply(_subject_snr, include_groups=False)
    .reset_index()
    .rename(columns={0: "snr"})
)

result_snr = mwu_family(subj_snr, "snr", correction="bonferroni")
print_bottom_line(result_snr)
result_snr

  v1 vs v2: U=95.0  p_corrected=0.2794 (not significant, p_raw=0.2794)  r=-0.267


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v1 vs v2,15,10,1.495,1.256,1.438,1.332,0.339,0.215,95.0,0.2794,0.2794,-0.267


### Test 6 — Reliability across v3+ minor versions

Per-subject median test-retest Spearman r (the `reliability` column, precomputed by
`parser_v2`), compared pairwise across v3.0 / v3.06 / v4.0. No formal statistical comparison of
reliability existed before this — only the visual `fig_reliability` above.

In [20]:
subj_reliability = (
    df_trials[df_trials["reliability"].notna()]
    .groupby(["participant_id", "task_version"])["reliability"]
    .median()
    .reset_index()
)
subj_reliability_v3plus = subj_reliability[subj_reliability["task_version"] >= 3.0]

result_reliability = mwu_family(subj_reliability_v3plus, "reliability", correction="bonferroni")
print_bottom_line(result_reliability)
result_reliability

  v3 vs v3.06: U=36.0  p_corrected=0.3451 (not significant, p_raw=0.1150)  r=0.405
  v3 vs v4: U=118.0  p_corrected=0.5137 (not significant, p_raw=0.1712)  r=0.285
  v3.06 vs v4: U=196.0  p_corrected=1.0000 (not significant, p_raw=0.3695)  r=-0.188


,N (A),N (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p_raw,r
comparison,,,,,,,,,,,,
v3 vs v3.06,11,11,0.206,0.286,0.208,0.334,0.156,0.219,36.0,0.3451,0.1150,0.405
v3 vs v4,11,30,0.206,0.256,0.208,0.268,0.156,0.153,118.0,0.5137,0.1712,0.285
v3.06 vs v4,11,30,0.286,0.256,0.334,0.268,0.219,0.153,196.0,1.0000,0.3695,-0.188


## 12. Quality Control

Validate the catch-trial QC thresholds in `SpAM_Task/task_config.json` against actual collected
data. `computeCatchQcFlag` (`utils.js`) flags a catch trial if **any** of:

- `cluster_mean_distance > catch_trials.cluster_max_mean` (0.15) — images not clustered tightly
- `cluster_sd > catch_trials.cluster_max_sd` (0.10) — cluster too spread (SD of pairwise distances; not a stored field, recomputed below)
- `max_dist_to_target > catch_trials.location_tolerance` (0.20) — worst-case per-image distance to the target corner/centre (mirrors `allImagesNearTarget`; not a stored field, recomputed below)

`cluster_mean_distance` is a stored field; `cluster_sd` and `max_dist_to_target` are recomputed
here from `pairwise_distances` / `final_locations` since the task only saves the boolean QC
outcome, not these intermediate values.

**Note**: the `qc_flag` column reflects whichever thresholds were active in `task_config.json`
*at the time each session ran* — which may differ from the current values above (config has
changed over time). The recomputed metrics below are compared against **today's** thresholds,
which is what answers "are these values still valid going forward."

In [21]:
df_catch = df_trials[df_trials["is_catch"]]

# Current catch_trials QC thresholds (SpAM_Task/task_config.json)
_CATCH_THRESH = {
    "cluster_mean_distance": 0.15,   # catch_trials.cluster_max_mean
    "cluster_sd":            0.10,   # catch_trials.cluster_max_sd
    "max_dist_to_target":    0.20,   # catch_trials.location_tolerance
}

_EDGE = 0.15  # fraction from edge for corner targets (utils.js _targetPoint)
_TARGET_FRAC = {
    "center":              (0.50, 0.50),
    "top left corner":     (_EDGE, _EDGE),
    "top right corner":    (1 - _EDGE, _EDGE),
    "bottom left corner":  (_EDGE, 1 - _EDGE),
    "bottom right corner": (1 - _EDGE, 1 - _EDGE),
}


def _cluster_sd(pw_json):
    """Sample SD (ddof=1) of a trial's pairwise distances -- mirrors computeSD() in utils.js.
    Generic across catch and main trials (same pairwise_distances schema)."""
    dists = list(parse_pairwise_distances(pw_json).values())
    return float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0


def _max_dist_to_target(row):
    """Worst-case (max) per-image normalised distance to the catch target -- mirrors allImagesNearTarget() in utils.js."""
    try:
        locs = json.loads(row["final_locations"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    if not locs:
        return np.nan
    fx, fy = _TARGET_FRAC.get(row["catch_trial_target_location"], (0.5, 0.5))
    w, h = row["sort_area_width"], row["sort_area_height"]
    diag = np.sqrt(w**2 + h**2)
    dists = [
        np.sqrt((loc["x"] * w - fx * w) ** 2 + (loc["y"] * h - fy * h) ** 2) / diag
        for loc in locs
    ]
    return max(dists)


df_catch = df_catch.assign(
    cluster_sd=df_catch["pairwise_distances"].apply(_cluster_sd),
    max_dist_to_target=df_catch.apply(_max_dist_to_target, axis=1),
)


def _qc_row(df, thresh_map, direction="max"):
    """
    Per-cohort QC summary row.

    direction="max": metric is capped (flag if value > thresh); shows median/p90/max
                      -- the upper tail is what's relevant for headroom.
    direction="min": metric is floored (flag if value < thresh); shows median/p10/min
                      -- the lower tail is what's relevant for headroom.
    """
    out = {}
    for metric, thresh in thresh_map.items():
        vals = df[metric].dropna()
        if direction == "max":
            out[(metric, "median")]               = round(float(vals.median()), 3)
            out[(metric, "p90")]                  = round(float(vals.quantile(0.9)), 3)
            out[(metric, "max")]                  = round(float(vals.max()), 3)
            out[(metric, f"flagged (>{thresh})")] = round(float((vals > thresh).mean()), 3)
        else:
            out[(metric, "median")]                = round(float(vals.median()), 3)
            out[(metric, "p10")]                   = round(float(vals.quantile(0.1)), 3)
            out[(metric, "min")]                   = round(float(vals.min()), 3)
            out[(metric, f"flagged (<{thresh})")]  = round(float((vals < thresh).mean()), 3)
    out[("recorded", "qc_flag rate")] = round(float(df["qc_flag"].mean()), 3)
    out[("recorded", "n_trials")]     = len(df)
    return pd.Series(out)


rows = {
    f"v{v:g}": _qc_row(df_catch[df_catch["task_version"] == v], _CATCH_THRESH)
    for v in sorted(df_catch["task_version"].unique())
}
rows["all"] = _qc_row(df_catch, _CATCH_THRESH)

qc_table = pd.DataFrame(rows).T
qc_table.columns = pd.MultiIndex.from_tuples(qc_table.columns)
qc_table

cluster_mean_distance                               cluster_sd         \
                     median    p90    max flagged (>0.15)     median    p90   
v1                    0.045  0.114  0.139             0.0      0.023  0.050   
v2                    0.027  0.046  0.073             0.0      0.014  0.025   
v3                    0.045  0.092  0.113             0.0      0.021  0.041   
v3.06                 0.030  0.094  0.142             0.0      0.013  0.040   
v4                    0.019  0.037  0.120             0.0      0.009  0.018   
all                   0.025  0.078  0.142             0.0      0.013  0.037   

                            max_dist_to_target                               \
         max flagged (>0.1)             median    p90    max flagged (>0.2)   
v1     0.061            0.0              0.115  0.164  0.197            0.0   
v2     0.029            0.0              0.119  0.156  0.183            0.0   
v3     0.049            0.0              0.128  0.162  0.173            0.0   
v3.06  0.079            0.0              0.132  0.173  0.199            0.0   
v4     0.051            0.0              0.093  0.153  0.170            0.0   
all    0.079            0.0              0.119  0.162  0.199            0.0   

          recorded           
      qc_flag rate n_trials  
v1           0.000     30.0  
v2           1.000     20.0  
v3           0.000     44.0  
v3.06        0.000     44.0  
v4           0.000     86.0  
all          0.089    224.0

### Experimental (main) trial thresholds

`computeMainQcFlag` (`utils.js`) flags a main trial if **either**:

- `pairwise_sd < quality_control.min_pairwise_distance_sd` (0.04) — images piled in one spot
- `move_ratio < quality_control.min_move_item_ratio` (0.75), where `move_ratio = num_moves / n_items` — too few moves relative to the number of images on screen

`quality_control.min_trial_rt_ms` (60 s) is excluded here since it's UI-enforced (the Done button
is disabled until the floor elapses), not a post-hoc statistical flag.

Both remaining checks are **floors**, the opposite direction from the catch-trial checks above, so
the table below shows the lower tail (median / p10 / min) and flags values *below* threshold.

In [22]:
# Current main-trial QC thresholds (SpAM_Task/task_config.json)
_MAIN_THRESH = {
    "pairwise_sd": 0.04,   # quality_control.min_pairwise_distance_sd
    "move_ratio":  0.75,   # quality_control.min_move_item_ratio
}


def _n_items(final_locations_json):
    try:
        return len(json.loads(final_locations_json))
    except (json.JSONDecodeError, TypeError):
        return np.nan


df_trials_qc = df_trials[~df_trials["is_catch"]].assign(
    pairwise_sd=lambda d: d["pairwise_distances"].apply(_cluster_sd),
    n_items=lambda d: d["final_locations"].apply(_n_items),
)
df_trials_qc["move_ratio"] = df_trials_qc["num_moves"] / df_trials_qc["n_items"]

rows = {
    f"v{v:g}": _qc_row(df_trials_qc[df_trials_qc["task_version"] == v], _MAIN_THRESH, direction="min")
    for v in sorted(df_trials_qc["task_version"].unique())
}
rows["all"] = _qc_row(df_trials_qc, _MAIN_THRESH, direction="min")

qc_table_main = pd.DataFrame(rows).T
qc_table_main.columns = pd.MultiIndex.from_tuples(qc_table_main.columns)
qc_table_main

pairwise_sd                               move_ratio               \
           median    p10    min flagged (<0.04)     median    p10   min   
v1          0.186  0.156  0.098             0.0      1.300  0.795  0.45   
v2          0.173  0.144  0.094             0.0      1.350  0.945  0.70   
v3          0.182  0.138  0.088             0.0      1.325  0.750  0.25   
v3.06       0.181  0.130  0.083             0.0      1.150  0.700  0.25   
v4          0.189  0.161  0.087             0.0      1.300  0.800  0.40   
all         0.185  0.151  0.083             0.0      1.300  0.750  0.25   

                          recorded           
      flagged (<0.75) qc_flag rate n_trials  
v1              0.060        0.000    150.0  
v2              0.010        0.000    100.0  
v3              0.091        0.091    220.0  
v3.06           0.132        0.132    220.0  
v4              0.063        0.024    632.0  
all             0.075        0.048   1322.0

### v4.0+ screening pass/fail summary

Quick view of the screening-block outcome, now that `parser_v2` preserves the screening_eval
diagnostic fields on `df_participants` (`reasons`, `move_ratio_fail_rate`,
`distance_sd_fail_rate`, `min_reliability`, `median_reliability`).

In [23]:
df_v4 = df_participants[df_participants["task_version"] >= 4.0]

screening_summary = (
    df_v4
    .groupby("task_version")
    .agg(
        n=("participant_id", "count"),
        n_screened_out=("status", lambda s: int((s == "screened out").sum())),
        n_full_data=("status", lambda s: int((s == "full data").sum())),
        screened_out_rate=("status", lambda s: round(float((s == "screened out").mean()), 3)),
    )
)
screening_summary

,n,n_screened_out,n_full_data,screened_out_rate
task_version,,,,
4.0,31,2,28,0.065
